# TTA Param Sweep: Best-4 Ensemble

Notebook này chạy sweep hệ số ensemble cho 4 cấu hình mạnh đã chọn:

- `dota_bw0.55_m0.95_ct0.9`
- `free_bw0.6_m0.9_p1`
- `bca_t0.03_bw0.7_pm0.95_xm0.98`
- `tda_pa0.4_pb5.5_pe0.4_na0`

Method `best4_ensemble` blend probability của 4 adapter theo weight-set `dota,free,bca,tda` và tự normalize tổng hệ số.


## Kaggle Setup


In [ ]:
!git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
%cd /kaggle/working/training-free-tta-for-deepfake-detection
!pip install -q -e . --no-deps


## Check Inputs


In [ ]:
!ls -lah /kaggle/input
!find /kaggle/input -maxdepth 3 -type f \( -name "*.pt" -o -name "*.csv" \) | sort | sed -n "1,180p"


## Method Availability


In [ ]:
REQUESTED_METHODS = {"Best-4 ensemble": "best4_ensemble"}
METHODS_TO_SWEEP = ["none", "best4_ensemble"]

print("Sweeping:", METHODS_TO_SWEEP)
print("Ensemble components:")
print("- dota_bw0.55_m0.95_ct0.9")
print("- free_bw0.6_m0.9_p1")
print("- bca_t0.03_bw0.7_pm0.95_xm0.98")
print("- tda_pa0.4_pb5.5_pe0.4_na0")


## Run Ensemble Weight Sweep

Cell này sweep `none + best4_ensemble`. Mỗi weight-set có dạng `dota,free,bca,tda`; script tự normalize nên hệ số không bắt buộc cộng đúng 1.


In [ ]:
FFPP_SPLIT = "/kaggle/input/ffpp-split-features"
FFPP_CORR = "/kaggle/input/ffpp-test-embeddings/ffpp_test_embeddings"
CELEB_CORR = "/kaggle/input/deepfakebench-features"
MODELS = "/kaggle/input/ffpp-training-free-models"
MODELS_BAL = "/kaggle/input/ffpp-training-free-models-balanced"

# Format: dota,free,bca,tda. Script tự normalize nên có thể dùng 0 để ablate component.
ENSEMBLE_WEIGHT_SETS = [
    # single-method sanity checks
    "1.00,0.00,0.00,0.00",
    "0.00,1.00,0.00,0.00",
    "0.00,0.00,1.00,0.00",
    "0.00,0.00,0.00,1.00",

    # pair ensembles
    "0.50,0.50,0.00,0.00",
    "0.50,0.00,0.50,0.00",
    "0.50,0.00,0.00,0.50",
    "0.00,0.50,0.50,0.00",
    "0.00,0.50,0.00,0.50",
    "0.00,0.00,0.50,0.50",

    # three-method ensembles
    "0.34,0.33,0.33,0.00",
    "0.34,0.33,0.00,0.33",
    "0.34,0.00,0.33,0.33",
    "0.00,0.34,0.33,0.33",

    # four-method weighted sweeps
    "0.25,0.25,0.25,0.25",
    "0.40,0.20,0.25,0.15",
    "0.35,0.20,0.30,0.15",
    "0.30,0.20,0.35,0.15",
    "0.30,0.25,0.30,0.15",
    "0.25,0.20,0.40,0.15",
    "0.25,0.15,0.40,0.20",
    "0.20,0.20,0.40,0.20",
    "0.20,0.15,0.45,0.20",
    "0.15,0.20,0.45,0.20",
]

print("Sweeping:", METHODS_TO_SWEEP)
print("Weight sets:", ENSEMBLE_WEIGHT_SETS)


## Block Size

Dùng cho balanced-aligned corruption datasets.


In [ ]:
BLOCK_SIZE = 16


In [ ]:
METHOD_ARGS = " ".join(METHODS_TO_SWEEP)
ENSEMBLE_WEIGHT_ARGS = " ".join(ENSEMBLE_WEIGHT_SETS)

!python testing/evaluate_tta_param_sweep.py \
  --train-features {FFPP_SPLIT}/ffpp_train_features.pt \
  --dataset ffpp-test={FFPP_SPLIT}/ffpp_test_features.pt \
  --dataset ffpp-test-corruption={FFPP_CORR} \
  --dataset celebdfv1-test-corruption={CELEB_CORR} \
  --dataset ffpp-test-balanced={FFPP_SPLIT}/ffpp_test_features.pt \
  --dataset ffpp-test-corruption-balanced={FFPP_CORR} \
  --dataset celebdfv1-test-corruption-balanced={CELEB_CORR} \
  --balanced-dataset ffpp-test-balanced \
  --balanced-aligned-dataset ffpp-test-corruption-balanced \
  --balanced-aligned-dataset celebdfv1-test-corruption-balanced \
  --block-size {BLOCK_SIZE} \
  --model linear-probe={MODELS}/ffpp_linear_probe_split.pt \
  --model osd={MODELS}/ffpp_osd_linear_probe_split.pt \
  --model linear-probe-balanced={MODELS_BAL}/ffpp_linear_probe_split.pt \
  --model osd-balanced={MODELS_BAL}/ffpp_osd_linear_probe_split.pt \
  --thresholds-csv linear-probe={MODELS}/thresholds.csv \
  --thresholds-csv osd={MODELS}/thresholds.csv \
  --thresholds-csv linear-probe-balanced={MODELS_BAL}/thresholds.csv \
  --thresholds-csv osd-balanced={MODELS_BAL}/thresholds.csv \
  --methods {METHOD_ARGS} \
  --ensemble-weights {ENSEMBLE_WEIGHT_ARGS} \
  --balance-method-fit \
  --continue-on-error \
  --results-output /kaggle/working/tta_best4_ensemble_sweep_results.csv


## Preview Results


In [ ]:
import pandas as pd

results = pd.read_csv("/kaggle/working/tta_best4_ensemble_sweep_results.csv")
display(results.head())
display(results.sort_values(["dataset", "method", "f1"], ascending=[True, True, False]).head(40))


## Best Params


In [ ]:
metric = "f1"
valid = results[results["error"].isna()] if "error" in results.columns else results
metric_cols = ["acc", "f1", "auc", "ap", "eer"]
summary = valid.groupby(["dataset", "model", "method", "param_id"], dropna=False)[metric_cols].mean().reset_index()
best = (summary.sort_values(["dataset", "model", "method", metric], ascending=[True, True, True, False])
        .groupby(["dataset", "model", "method"], as_index=False)
        .head(3))
summary.to_csv("/kaggle/working/tta_best4_ensemble_sweep_summary.csv", index=False)
best.to_csv("/kaggle/working/tta_best4_ensemble_sweep_best.csv", index=False)
display(best)


## Notes

Ensemble này chạy online từng component theo cùng thứ tự test batch rồi blend xác suất cuối cùng. Vì các component có state online riêng, thay đổi batch size hoặc shuffle có thể làm kết quả đổi nhẹ.
